# Fine-tuning modèle médical — TechCorp Hackathon
**Groupe 23 — DATA/IA**

**Avant de lancer :**
1. `Exécution` > `Modifier le type d'exécution` > choisir **GPU T4**
2. `Exécution` > `Tout exécuter`

In [ ]:
# Vérification GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Pas de GPU — aller dans Exécution > Modifier le type execution > GPU T4')

In [ ]:
# Installation Unsloth (compatible Colab, pas de problème bitsandbytes)
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate datasets
print('Installation terminée !')

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'Aucun GPU détecté !')
print('VRAM :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

## 1. Chargement et nettoyage du dataset médical

In [ ]:
print('Chargement du dataset médical...')
raw_dataset = load_dataset('ruslanmv/ai-medical-chatbot', split='train')
print(f'Dataset brut : {len(raw_dataset)} exemples')

In [ ]:
# Nettoyage
def nettoyer(dataset):
    cleaned = []
    seen = set()
    for ex in dataset:
        patient = (ex.get('Patient') or '').strip()
        doctor  = (ex.get('Doctor')  or '').strip()
        if doctor.endswith('-->'): continue
        if len(patient) < 10 or len(doctor) < 10: continue
        key = (patient[:80], doctor[:80])
        if key in seen: continue
        seen.add(key)
        text = f"<|user|>\n{patient}<|end|>\n<|assistant|>\n{doctor}<|end|>"
        cleaned.append({'text': text})
    return cleaned

data = nettoyer(raw_dataset)
dataset = Dataset.from_list(data).shuffle(seed=42).select(range(min(20000, len(data))))
print(f'Après nettoyage : {len(dataset)} exemples')
print('\nExemple :')
print(dataset[0]['text'][:400])

## 2. Chargement du modèle avec Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Phi-3-mini-4k-instruct',
    max_seq_length=512,
    dtype=None,
    load_in_4bit=True,
)
print('Modèle chargé !')

## 3. Configuration LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=['qkv_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    use_gradient_checkpointing=True,
    random_state=42,
)
model.print_trainable_parameters()

## 4. Entraînement

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=512,
    args=TrainingArguments(
        output_dir='./medical_model_trained',
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        warmup_steps=50,
        logging_steps=25,
        save_steps=500,
        save_total_limit=1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        remove_unused_columns=False,
        report_to='none',
    ),
)

print("Lancement de l'entraînement...")
result = trainer.train()

print('\n=== METRIQUES ===')
print(f'Loss finale  : {result.training_loss:.4f}')
print(f'Epochs       : 3')
print(f'Exemples     : {len(dataset)}')
print(f'Durée        : {round(result.metrics["train_runtime"] / 60, 1)} minutes')

## 5. Test du modèle

In [ ]:
FastLanguageModel.for_inference(model)

def repondre(question):
    prompt = f'<|user|>\n{question}<|end|>\n<|assistant|>\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

questions = [
    'I have a headache and fever for 3 days, what should I do?',
    'What are the symptoms of diabetes?',
    'I feel very tired all the time, could it be anemia?'
]

print('=== TEST DU MODELE ===')
for q in questions:
    print(f'\nQuestion : {q}')
    print(f'Réponse  : {repondre(q)}')
    print('-' * 60)

In [ ]:
# Sauvegarde
trainer.save_model('./medical_model_trained')
print('Modèle sauvegardé !')